# API Model Registration Demo

This notebook walks through registering a model that is served via an external PyMC API.
The API is packaged as a Docker service (`pymc_api`) which exposes a synchronous `/runs`
endpoint. The Model Zoo registry stores the API binding and uses it when executing runs.

## 1. Start the Docker services

The command below launches Postgres, the registry API, and the dummy PyMC service.
Skip this step if you already have them running.

In [ ]:
%%bash
set -euxo pipefail
docker compose up -d postgres api pymc_api
docker compose ps

## 2. Inspect the API-backed configuration

The configuration declares `kind: "api"` and binds to the service endpoint.

In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'configs').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

config_path = (PROJECT_ROOT / 'configs' / 'pymc_api_linear.json').resolve()
api_config = json.loads(config_path.read_text())
config_path, api_config


## 3. Register the configuration

POST the config to the registry. The binding is stored with the model spec so future runs
can resolve the API endpoint automatically.

In [ ]:
import requests

registry_url = 'http://localhost:8000/registry/configs'
response = requests.post(registry_url, json={'env': 'demo-api', 'config': api_config}, timeout=30)
response.raise_for_status()
response.json()

## 4. Submit a run

Synthetic data is sent to the registry. It forwards the inputs to the PyMC API
and returns the predictions and diagnostics.

In [ ]:
import numpy as np

x_values = (np.linspace(0.0, 4.5, num=25) + np.random.normal(0, 0.1, size=25)).round(3).tolist()
y_values = [1.5 + 2.2 * x for x in x_values]

run_payload = {
    'model_id': api_config['model']['id'],
    'env': 'demo-api',
    'data_sources': {'x_values': x_values, 'y_values': y_values},
    'context': {'timeout_s': 120}
}

submit_resp = requests.post('http://localhost:8000/runs', json=run_payload, timeout=30)
submit_resp.raise_for_status()
ticket = submit_resp.json()
ticket

## 5. Poll the run result

The API-backed run returns almost immediately. Inspect the summary returned by the PyMC service.

In [ ]:
run_id = ticket['run_id']
status_url = f'http://localhost:8000/runs/{run_id}'
result_resp = requests.get(status_url, timeout=30)
result_resp.raise_for_status()
result = result_resp.json()
result['result']['summary']
